
# Notebook 04 — Mathematical Supply Network Optimization

## Objective
Develop a time-phased mathematical optimization
model that minimizes total hardware supply
network operating costs.

## Methodology
- Python
- Pyomo mathematical modeling
- HiGHS optimization solver
- Linear programming

## Decision Variables
1. Supplier purchase quantities
2. Manufacturing quantities
3. Transportation shipment quantities
4. Component inventory levels
5. Plant finished goods inventory
6. Distribution center inventory levels

## Business Constraints
- Supplier capacity
- Supplier procurement lead times
- BOM component consumption
- Manufacturing capacity
- Transportation lane capacity
- Transportation transit lead times
- Distribution center storage capacity
- Regional customer demand satisfaction

## Comparison Requirements
The optimizer will use the same demand, initial
inventory, unit costs, and operating constraints
as Notebook 03.

It will require full customer demand fulfillment
and matching final inventory quantities for a
fair baseline comparison.

## Expected Deliverables
- Optimized weekly supply plan
- Optimized total operating cost
- Baseline versus optimization comparison
- Source allocation decisions
- Manufacturing and shipment schedules
- Inventory and service performance


In [1]:
# STEP 2: Initialize mathematical optimization notebook

import pandas as pd
import numpy as np
import pyomo.environ as pyo
from pathlib import Path

# Locate Project 2 repository
current_path = Path.cwd().resolve()

repo_root = next(
    (
        path for path in [current_path, *current_path.parents]
        if (path / "data/assumptions/bom.csv").is_file()
    ),
    None
)

assert repo_root is not None, "Project 2 repository not found"

# Load network assumptions and demand
input_paths = {
    "bom": "data/assumptions/bom.csv",
    "suppliers": "data/assumptions/suppliers.csv",
    "plants": "data/assumptions/plants.csv",
    "dcs": "data/assumptions/distribution_centers.csv",
    "transport": "data/assumptions/transport_lanes.csv",
    "inventory": "data/assumptions/initial_inventory.csv",
    "allocation": "data/assumptions/market_allocation.csv",
    "demand": "data/processed/demand_plan_12_weeks.csv",
    "regional": "data/processed/regional_demand_12_weeks.csv"
}

inputs = {
    name: pd.read_csv(repo_root / path)
    for name, path in input_paths.items()
}

# Load baseline performance
baseline_kpis = pd.read_csv(
    repo_root / "outputs/tables/baseline_kpis.csv"
)

baseline_costs = pd.read_csv(
    repo_root / "outputs/tables/baseline_weekly_costs.csv"
)

baseline_dc_inventory = pd.read_csv(
    repo_root / "outputs/tables/baseline_dc_inventory.csv"
)

baseline_component_inventory = pd.read_csv(
    repo_root / "outputs/tables/baseline_component_inventory.csv"
)

baseline_plant_inventory = pd.read_csv(
    repo_root / "outputs/tables/baseline_plant_inventory.csv"
)

# Validate baseline cost
baseline_total_cost = baseline_costs[
    "Total_Cost_USD"
].sum()

assert baseline_total_cost > 0

assert np.isclose(
    inputs["regional"]["Demand_Units"].sum(),
    6180
)

print("OPTIMIZATION ENVIRONMENT READY")
print("=" * 45)
print("Network datasets:", len(inputs))
print("Planning weeks:", len(inputs["demand"]))
print("Baseline cost: $", round(baseline_total_cost, 2))
print("Pyomo imported successfully!")

display(baseline_kpis)


OPTIMIZATION ENVIRONMENT READY
Network datasets: 9
Planning weeks: 12
Baseline cost: $ 414139.7
Pyomo imported successfully!


,Metric,Value,Unit
0,Total Demand,6180.000000,Units
1,Fulfilled Demand,6180.000000,Units
2,Shortages,0.000000,Units
3,Service Level,100.000000,%
4,Total Production,5665.000000,Units
5,Plant Utilization,81.393678,%
6,Expedited Freight Share,40.000000,%
7,Average Ending DC Inventory,45.000000,Units
8,Final DC Inventory,45.000000,Units
9,Total Modeled Cost,414139.700000,USD


In [2]:
# STEP 3: Test Pyomo and HiGHS solver

# Create a small test model
test_model = pyo.ConcreteModel()

# Decision variables
test_model.x = pyo.Var(domain=pyo.NonNegativeReals)
test_model.y = pyo.Var(domain=pyo.NonNegativeReals)

# Minimize cost
test_model.objective = pyo.Objective(
    expr=8 * test_model.x + 11 * test_model.y,
    sense=pyo.minimize
)

# Demand requirement
test_model.demand = pyo.Constraint(
    expr=test_model.x + test_model.y == 1200
)

# Supplier capacities
test_model.capacity_a = pyo.Constraint(
    expr=test_model.x <= 800
)

test_model.capacity_b = pyo.Constraint(
    expr=test_model.y <= 600
)

# Initialize HiGHS solver
solver = pyo.SolverFactory("highs")

assert solver.available(
    exception_flag=False
), "HiGHS solver is not available"

# Solve test problem
results = solver.solve(test_model)

assert (
    results.solver.termination_condition
    == pyo.TerminationCondition.optimal
)

print("OPTIMIZATION SOLVER TEST")
print("=" * 40)

print("Solver: HiGHS")
print("Status:", results.solver.termination_condition)

print(
    "Supplier A:",
    pyo.value(test_model.x)
)

print(
    "Supplier B:",
    pyo.value(test_model.y)
)

print(
    "Minimum cost: $",
    pyo.value(test_model.objective)
)

print("\nSOLVER TEST: PASSED")


OPTIMIZATION SOLVER TEST
Solver: HiGHS
Status: optimal
Supplier A: 800.0
Supplier B: 400.0
Minimum cost: $ 10800.0

SOLVER TEST: PASSED


In [3]:
# STEP 4B: Prepare model sets and parameters

# Retrieve validated data
bom_df = inputs["bom"]
suppliers_df = inputs["suppliers"]
plants_df = inputs["plants"]
dcs_df = inputs["dcs"]
lanes_df = inputs["transport"]
inventory_df = inputs["inventory"]
demand_df = inputs["regional"]

# Define planning horizon
weeks = list(range(1, 13))

# Define supply network entities
components = bom_df["Component_ID"].unique().tolist()
suppliers = suppliers_df["Supplier_ID"].unique().tolist()
distribution_centers = dcs_df["DC_ID"].unique().tolist()
lanes = lanes_df["Lane_ID"].unique().tolist()

# Eligible supplier-component combinations
eligible_suppliers = suppliers_df.loc[
    suppliers_df["Eligible"] == True
].copy()

supplier_component_pairs = list(
    eligible_suppliers[
        ["Supplier_ID", "Component_ID"]
    ].itertuples(index=False, name=None)
)

# Weekly demand by distribution center
regional_demand = {
    (row.DC_ID, int(row.Week)): float(row.Demand_Units)
    for row in demand_df.itertuples(index=False)
}

# BOM quantity requirements
bom_quantity = {
    row.Component_ID: float(row.Units_Per_Finished_Unit)
    for row in bom_df.itertuples(index=False)
}

# Supplier procurement parameters
supplier_cost = {
    (row.Supplier_ID, row.Component_ID):
        float(row.Unit_Cost_USD)
    for row in eligible_suppliers.itertuples(index=False)
}

supplier_capacity = {
    (row.Supplier_ID, row.Component_ID):
        float(row.Weekly_Capacity_Units)
    for row in eligible_suppliers.itertuples(index=False)
}

supplier_lead_time = {
    (row.Supplier_ID, row.Component_ID):
        int(row.Lead_Time_Weeks)
    for row in eligible_suppliers.itertuples(index=False)
}

# Transportation parameters
lane_cost = dict(zip(
    lanes_df["Lane_ID"],
    lanes_df["Cost_Per_Unit_USD"]
))

lane_capacity = dict(zip(
    lanes_df["Lane_ID"],
    lanes_df["Weekly_Lane_Capacity_Units"]
))

lane_lead_time = dict(zip(
    lanes_df["Lane_ID"],
    lanes_df["Transit_Time_Weeks"]
))

lane_destination = dict(zip(
    lanes_df["Lane_ID"],
    lanes_df["Destination_DC_ID"]
))

# Distribution center parameters
dc_capacity = dict(zip(
    dcs_df["DC_ID"],
    dcs_df["Storage_Capacity_Units"]
))

dc_holding_cost = dict(zip(
    dcs_df["DC_ID"],
    dcs_df["Holding_Cost_Per_Unit_Per_Week_USD"]
))

# Manufacturing parameters
assert len(plants_df) == 1

plant_capacity = float(
    plants_df["Weekly_Capacity_Units"].iloc[0]
)

assembly_cost = float(
    plants_df["Assembly_Cost_Per_Unit_USD"].iloc[0]
)

# Initial component inventory
initial_component_inventory = {
    row.Item_ID: float(row.Beginning_Inventory_Units)
    for row in inventory_df.loc[
        inventory_df["Item_Type"] == "Component"
    ].itertuples(index=False)
}

# Initial finished goods inventory by location
finished_inventory_df = inventory_df.loc[
    inventory_df["Item_Type"] == "Finished_Good"
]

initial_dc_inventory = {
    dc: float(finished_inventory_df.loc[
        finished_inventory_df["Location_ID"] == dc,
        "Beginning_Inventory_Units"
    ].iloc[0])
    for dc in distribution_centers
}

initial_plant_inventory = float(
    finished_inventory_df.loc[
        finished_inventory_df["Location_ID"] ==
        plants_df["Plant_ID"].iloc[0],
        "Beginning_Inventory_Units"
    ].iloc[0]
)

print("Optimization parameters prepared!")
print("Weeks:", len(weeks))
print("Components:", len(components))
print("Suppliers:", len(suppliers))
print("Distribution centers:", len(distribution_centers))
print("Transportation lanes:", len(lanes))
print("Supplier-component pairs:", len(supplier_component_pairs))
print("Total demand:", sum(regional_demand.values()))


Optimization parameters prepared!
Weeks: 12
Components: 3
Suppliers: 2
Distribution centers: 2
Transportation lanes: 4
Supplier-component pairs: 6
Total demand: 6180.0


In [4]:
# STEP 4C: Initialize mathematical optimization model

import pyomo.environ as pyo

model = pyo.ConcreteModel(
    name="Hardware_Supply_Network_Optimizer"
)

# Planning weeks
model.T = pyo.Set(
    initialize=weeks,
    ordered=True
)

# Hardware components
model.C = pyo.Set(
    initialize=components
)

# Suppliers
model.S = pyo.Set(
    initialize=suppliers
)

# Distribution centers
model.D = pyo.Set(
    initialize=distribution_centers
)

# Transportation lanes
model.L = pyo.Set(
    initialize=lanes
)

# Eligible supplier-component combinations
model.SC = pyo.Set(
    dimen=2,
    initialize=supplier_component_pairs
)

# Demand parameters
model.demand = pyo.Param(
    model.D,
    model.T,
    initialize=regional_demand,
    within=pyo.NonNegativeReals
)

# Component requirements
model.bom = pyo.Param(
    model.C,
    initialize=bom_quantity,
    within=pyo.PositiveReals
)

# Supplier parameters
model.supplier_cost = pyo.Param(
    model.SC,
    initialize=supplier_cost,
    within=pyo.NonNegativeReals
)

model.supplier_capacity = pyo.Param(
    model.SC,
    initialize=supplier_capacity,
    within=pyo.NonNegativeReals
)

model.supplier_lead_time = pyo.Param(
    model.SC,
    initialize=supplier_lead_time,
    within=pyo.NonNegativeIntegers
)

# Transportation parameters
model.lane_cost = pyo.Param(
    model.L,
    initialize=lane_cost,
    within=pyo.NonNegativeReals
)

model.lane_capacity = pyo.Param(
    model.L,
    initialize=lane_capacity,
    within=pyo.NonNegativeReals
)

model.lane_lead_time = pyo.Param(
    model.L,
    initialize=lane_lead_time,
    within=pyo.NonNegativeIntegers
)

# Distribution center parameters
model.dc_capacity = pyo.Param(
    model.D,
    initialize=dc_capacity,
    within=pyo.NonNegativeReals
)

model.dc_holding_cost = pyo.Param(
    model.D,
    initialize=dc_holding_cost,
    within=pyo.NonNegativeReals
)

# Manufacturing parameters
model.plant_capacity = pyo.Param(
    initialize=plant_capacity
)

model.assembly_cost = pyo.Param(
    initialize=assembly_cost
)

print("PYOMO MODEL INITIALIZED")
print("=" * 45)

print("Planning weeks:", len(model.T))
print("Components:", len(model.C))
print("Suppliers:", len(model.S))
print("Distribution centers:", len(model.D))
print("Transportation lanes:", len(model.L))
print("Supplier-component pairs:", len(model.SC))

print("\nMODEL INITIALIZATION: PASSED")


PYOMO MODEL INITIALIZED
Planning weeks: 12
Components: 3
Suppliers: 2
Distribution centers: 2
Transportation lanes: 4
Supplier-component pairs: 6

MODEL INITIALIZATION: PASSED


In [5]:

# STEP 4D: Validate optimization model inputs

# Validate set sizes
assert len(model.T) == 12
assert len(model.C) == 3
assert len(model.S) == 2
assert len(model.D) == 2
assert len(model.L) == 4
assert len(model.SC) == 6

# Validate demand coverage
assert len(regional_demand) == (
    len(model.T) * len(model.D)
)

total_model_demand = sum(
    pyo.value(model.demand[d, t])
    for d in model.D
    for t in model.T
)

assert np.isclose(total_model_demand, 6180)

# Validate supplier capacities and costs
for supplier, component in model.SC:

    assert pyo.value(
        model.supplier_capacity[supplier, component]
    ) > 0

    assert pyo.value(
        model.supplier_cost[supplier, component]
    ) >= 0

# Validate transportation lane parameters
for lane in model.L:

    assert pyo.value(model.lane_capacity[lane]) > 0
    assert pyo.value(model.lane_cost[lane]) >= 0

    assert lane_destination[lane] in model.D

# Validate initial inventory coverage
assert set(initial_component_inventory) == set(model.C)
assert set(initial_dc_inventory) == set(model.D)

# Validate production capacity
assert pyo.value(model.plant_capacity) == 580

print("OPTIMIZATION MODEL VALIDATION")
print("=" * 45)

print("Model sets: PASSED")
print("Demand parameters: PASSED")
print("Supplier parameters: PASSED")
print("Transportation parameters: PASSED")
print("Initial inventory: PASSED")
print("Manufacturing capacity: PASSED")

print("\nSTEP 4 VALIDATION: PASSED")


OPTIMIZATION MODEL VALIDATION
Model sets: PASSED
Demand parameters: PASSED
Supplier parameters: PASSED
Transportation parameters: PASSED
Initial inventory: PASSED
Manufacturing capacity: PASSED

STEP 4 VALIDATION: PASSED


In [6]:
# STEP 5B: Define optimization decision variables

import pyomo.environ as pyo

# 1. Supplier purchasing decisions
# Units ordered from each eligible supplier-component
# combination during each planning week
model.purchase = pyo.Var(
    model.SC,
    model.T,
    domain=pyo.NonNegativeIntegers
)

# 2. Manufacturing production decisions
# Finished product units manufactured each week
model.production = pyo.Var(
    model.T,
    domain=pyo.NonNegativeIntegers
)

# 3. Transportation shipment decisions
# Finished product units shipped along each lane
# during each planning week
model.ship = pyo.Var(
    model.L,
    model.T,
    domain=pyo.NonNegativeIntegers
)

# 4. Component inventory decisions
# Ending component stock at the manufacturing plant
model.component_inventory = pyo.Var(
    model.C,
    model.T,
    domain=pyo.NonNegativeIntegers
)

# 5. Plant finished goods inventory decisions
# Ending finished products at the manufacturing plant
model.plant_inventory = pyo.Var(
    model.T,
    domain=pyo.NonNegativeIntegers
)

# 6. Distribution center inventory decisions
# Ending finished goods at each distribution center
model.dc_inventory = pyo.Var(
    model.D,
    model.T,
    domain=pyo.NonNegativeIntegers
)

print("OPTIMIZATION DECISION VARIABLES CREATED")
print("=" * 50)

print("Supplier purchasing:", len(model.purchase))
print("Manufacturing:", len(model.production))
print("Transportation:", len(model.ship))
print("Component inventory:", len(model.component_inventory))
print("Plant finished inventory:", len(model.plant_inventory))
print("DC inventory:", len(model.dc_inventory))

total_variables = (
    len(model.purchase)
    + len(model.production)
    + len(model.ship)
    + len(model.component_inventory)
    + len(model.plant_inventory)
    + len(model.dc_inventory)
)

print("\nTotal decision variables:", total_variables)
print("STEP 5B: PASSED")


OPTIMIZATION DECISION VARIABLES CREATED
Supplier purchasing: 72
Manufacturing: 12
Transportation: 48
Component inventory: 36
Plant finished inventory: 12
DC inventory: 24

Total decision variables: 204
STEP 5B: PASSED


In [7]:
# STEP 5C: Validate optimization decision variables

# Verify expected variable counts
assert len(model.purchase) == 72
assert len(model.production) == 12
assert len(model.ship) == 48
assert len(model.component_inventory) == 36
assert len(model.plant_inventory) == 12
assert len(model.dc_inventory) == 24

# Verify all six variable categories exist
variable_names = [
    "purchase",
    "production",
    "ship",
    "component_inventory",
    "plant_inventory",
    "dc_inventory"
]

for name in variable_names:
    variable = getattr(model, name)

    assert variable.is_indexed()

    for index in variable:
        assert variable[index].domain is (
            pyo.NonNegativeIntegers
        )

# Validate total variable count
total_variables = sum(
    len(getattr(model, name))
    for name in variable_names
)

assert total_variables == 204

# Confirm model structure
print("DECISION VARIABLE VALIDATION")
print("=" * 45)

print("Purchasing variables: PASSED")
print("Production variables: PASSED")
print("Transportation variables: PASSED")
print("Component inventory variables: PASSED")
print("Plant inventory variables: PASSED")
print("DC inventory variables: PASSED")

print("\nTotal variables:", total_variables)
print("STEP 5 VALIDATION: PASSED")


DECISION VARIABLE VALIDATION
Purchasing variables: PASSED
Production variables: PASSED
Transportation variables: PASSED
Component inventory variables: PASSED
Plant inventory variables: PASSED
DC inventory variables: PASSED

Total variables: 204
STEP 5 VALIDATION: PASSED


In [8]:
# STEP 6B: Define supply network cost expressions

import pyomo.environ as pyo

# 1. PROCUREMENT COST
# Supplier unit cost multiplied by purchase quantities
model.procurement_cost_expr = pyo.Expression(
    expr=sum(
        model.supplier_cost[s, c]
        * model.purchase[s, c, t]
        for (s, c) in model.SC
        for t in model.T
    )
)

# 2. MANUFACTURING COST
# Assembly cost multiplied by production quantities
model.manufacturing_cost_expr = pyo.Expression(
    expr=sum(
        model.assembly_cost
        * model.production[t]
        for t in model.T
    )
)

# 3. TRANSPORTATION COST
# Freight cost multiplied by shipped quantities
model.freight_cost_expr = pyo.Expression(
    expr=sum(
        model.lane_cost[l]
        * model.ship[l, t]
        for l in model.L
        for t in model.T
    )
)

# 4. DISTRIBUTION INVENTORY HOLDING COST
# Weekly ending DC inventory multiplied by holding cost
model.holding_cost_expr = pyo.Expression(
    expr=sum(
        model.dc_holding_cost[d]
        * model.dc_inventory[d, t]
        for d in model.D
        for t in model.T
    )
)

print("SUPPLY NETWORK COST EXPRESSIONS CREATED")
print("=" * 45)
print("Procurement cost: DEFINED")
print("Manufacturing cost: DEFINED")
print("Transportation cost: DEFINED")
print("Inventory holding cost: DEFINED")
print("\nSTEP 6B: PASSED")


SUPPLY NETWORK COST EXPRESSIONS CREATED
Procurement cost: DEFINED
Manufacturing cost: DEFINED
Transportation cost: DEFINED
Inventory holding cost: DEFINED

STEP 6B: PASSED


In [9]:
# STEP 6C: Create cost-minimization objective

model.total_cost = pyo.Objective(
    expr=(
        model.procurement_cost_expr
        + model.manufacturing_cost_expr
        + model.freight_cost_expr
        + model.holding_cost_expr
    ),
    sense=pyo.minimize
)

print("COST-MINIMIZATION OBJECTIVE CREATED")
print("=" * 45)

print("Objective name:", model.total_cost.name)

print(
    "Optimization direction:",
    "MINIMIZE"
    if model.total_cost.sense == pyo.minimize
    else "MAXIMIZE"
)

print("STEP 6C: PASSED")


COST-MINIMIZATION OBJECTIVE CREATED
Objective name: total_cost
Optimization direction: MINIMIZE
STEP 6C: PASSED


In [10]:
# STEP 6D: Validate cost-minimization objective

from pyomo.repn.standard_repn import generate_standard_repn

# Verify objective exists and is active
assert model.total_cost.is_constructed()
assert model.total_cost.active

# Verify minimization direction
assert model.total_cost.sense == pyo.minimize

# Verify all four cost expressions exist
cost_expressions = [
    model.procurement_cost_expr,
    model.manufacturing_cost_expr,
    model.freight_cost_expr,
    model.holding_cost_expr
]

for expression in cost_expressions:
    assert expression.is_constructed()

# Verify objective is linear
representation = generate_standard_repn(
    model.total_cost.expr
)

assert representation.is_linear(), (
    "Objective contains nonlinear terms"
)

# Count objective variables
objective_variable_count = len(
    representation.linear_vars
)

print("COST OBJECTIVE VALIDATION")
print("=" * 45)

print("Objective constructed: PASSED")
print("Objective active: PASSED")
print("Minimization direction: PASSED")
print("Four cost categories: PASSED")
print("Linear objective: PASSED")

print(
    "\nVariables appearing in objective:",
    objective_variable_count
)

print("\nSTEP 6 VALIDATION: PASSED")


COST OBJECTIVE VALIDATION
Objective constructed: PASSED
Objective active: PASSED
Minimization direction: PASSED
Four cost categories: PASSED
Linear objective: PASSED

Variables appearing in objective: 156

STEP 6 VALIDATION: PASSED


In [11]:
# STEP 7B: Add supplier capacity constraints

import pyomo.environ as pyo

# Each supplier-component combination has
# a maximum purchasing capacity per week.

def supplier_capacity_rule(m, s, c, t):

    return (
        m.purchase[s, c, t]
        <= m.supplier_capacity[s, c]
    )

model.supplier_capacity_limit = pyo.Constraint(
    model.SC,
    model.T,
    rule=supplier_capacity_rule
)

print("SUPPLIER CAPACITY CONSTRAINTS CREATED")
print("=" * 45)

print(
    "Supplier capacity constraints:",
    len(model.supplier_capacity_limit)
)

assert len(model.supplier_capacity_limit) == 72

print("\nSTEP 7B: PASSED")


SUPPLIER CAPACITY CONSTRAINTS CREATED
Supplier capacity constraints: 72

STEP 7B: PASSED


In [12]:
# STEP 7C: Add component inventory balance constraints

# Beginning inventory was loaded in Step 4.
# Supplier receipts are determined by lead times.
# Production consumes components according to BOM.

first_week = min(weeks)

def component_balance_rule(m, c, t):

    # 1. Determine inventory available
    # from the previous period.
    if t == first_week:
        beginning = initial_component_inventory[c]
    else:
        beginning = m.component_inventory[c, t - 1]

    # 2. Calculate supplier orders arriving
    # during the current week.
    arrivals = 0

    for s, component in m.SC:

        if component != c:
            continue

        lead_time = int(
            pyo.value(m.supplier_lead_time[s, c])
        )

        order_week = t - lead_time

        # Only orders placed within the planning
        # horizon can generate receipts.
        if order_week in m.T:
            arrivals += m.purchase[s, c, order_week]

    # 3. Calculate component consumption
    # from manufacturing finished products.
    consumption = (
        m.bom[c] * m.production[t]
    )

    # 4. Enforce component inventory conservation.
    return (
        beginning
        + arrivals
        - consumption
        == m.component_inventory[c, t]
    )

model.component_inventory_balance = pyo.Constraint(
    model.C,
    model.T,
    rule=component_balance_rule
)

print("COMPONENT INVENTORY CONSTRAINTS CREATED")
print("=" * 45)

print(
    "Component inventory balance constraints:",
    len(model.component_inventory_balance)
)

assert len(model.component_inventory_balance) == 36

print("\nSTEP 7C: PASSED")


COMPONENT INVENTORY CONSTRAINTS CREATED
Component inventory balance constraints: 36

STEP 7C: PASSED


In [13]:
# STEP 7D: Prevent orders arriving after the planning horizon

last_week = max(weeks)

def purchase_horizon_rule(m, s, c, t):

    lead_time = int(
        pyo.value(m.supplier_lead_time[s, c])
    )

    arrival_week = t + lead_time

    # Orders arriving outside the planning
    # horizon are not allowed.
    if arrival_week > last_week:
        return m.purchase[s, c, t] == 0

    return pyo.Constraint.Skip

model.purchase_horizon_limit = pyo.Constraint(
    model.SC,
    model.T,
    rule=purchase_horizon_rule
)

print("PURCHASE HORIZON CONSTRAINTS CREATED")
print("=" * 45)

print(
    "Late-order restrictions:",
    len(model.purchase_horizon_limit)
)

assert len(model.purchase_horizon_limit) == 9

print("\nSTEP 7D: PASSED")


PURCHASE HORIZON CONSTRAINTS CREATED
Late-order restrictions: 9

STEP 7D: PASSED


In [14]:
# STEP 7E: Validate procurement and component constraints

from pyomo.repn.standard_repn import generate_standard_repn

# 1. Validate constraint counts
assert len(model.supplier_capacity_limit) == 72

assert len(model.component_inventory_balance) == 36

assert len(model.purchase_horizon_limit) == 9

# 2. Verify component balance equations are linear
for c in model.C:
    for t in model.T:

        constraint = model.component_inventory_balance[c, t]

        representation = generate_standard_repn(
            constraint.body
        )

        assert representation.is_linear(), (
            f"Nonlinear component balance: {c}, Week {t}"
        )

# 3. Verify supplier lead times
for s, c in model.SC:

    lead_time = int(
        pyo.value(model.supplier_lead_time[s, c])
    )

    assert lead_time >= 0

# 4. Verify initial component inventory
for c in model.C:

    assert initial_component_inventory[c] >= 0

    assert pyo.value(model.bom[c]) > 0

# 5. Calculate total constraints added in Step 7
step7_constraints = (
    len(model.supplier_capacity_limit)
    + len(model.component_inventory_balance)
    + len(model.purchase_horizon_limit)
)

assert step7_constraints == 117

print("STEP 7 CONSTRAINT VALIDATION")
print("=" * 45)

print("Supplier capacity constraints: PASSED")
print("Component inventory balances: PASSED")
print("Purchase horizon restrictions: PASSED")
print("Lead-time parameters: PASSED")
print("Initial inventory parameters: PASSED")
print("Constraint linearity: PASSED")

print("\nTotal Step 7 constraints:", step7_constraints)

print("\nSTEP 7 VALIDATION: PASSED")


STEP 7 CONSTRAINT VALIDATION
Supplier capacity constraints: PASSED
Component inventory balances: PASSED
Purchase horizon restrictions: PASSED
Lead-time parameters: PASSED
Initial inventory parameters: PASSED
Constraint linearity: PASSED

Total Step 7 constraints: 117

STEP 7 VALIDATION: PASSED


In [15]:
# STEP 8B: Add manufacturing capacity constraints

import pyomo.environ as pyo

# Production cannot exceed available
# manufacturing capacity in any planning week.

def manufacturing_capacity_rule(m, t):

    return (
        m.production[t]
        <= m.plant_capacity
    )

model.manufacturing_capacity_limit = pyo.Constraint(
    model.T,
    rule=manufacturing_capacity_rule
)

print("MANUFACTURING CAPACITY CONSTRAINTS CREATED")
print("=" * 45)

print(
    "Manufacturing capacity constraints:",
    len(model.manufacturing_capacity_limit)
)

assert len(model.manufacturing_capacity_limit) == 12

print("\nSTEP 8B: PASSED")


MANUFACTURING CAPACITY CONSTRAINTS CREATED
Manufacturing capacity constraints: 12

STEP 8B: PASSED


In [16]:
# STEP 8C: Add plant finished-goods inventory balances

first_week = min(model.T)

def plant_inventory_balance_rule(m, t):

    # 1. Determine beginning finished-goods inventory
    if t == first_week:
        beginning = initial_plant_inventory
    else:
        beginning = m.plant_inventory[t - 1]

    # 2. Finished products manufactured this week
    production = m.production[t]

    # 3. Total shipments departing the plant this week
    outbound_shipments = sum(
        m.ship[l, t]
        for l in m.L
    )

    # 4. Enforce plant finished-goods conservation
    return (
        beginning
        + production
        - outbound_shipments
        == m.plant_inventory[t]
    )

model.plant_inventory_balance = pyo.Constraint(
    model.T,
    rule=plant_inventory_balance_rule
)

print("PLANT INVENTORY CONSTRAINTS CREATED")
print("=" * 45)

print(
    "Plant inventory balance constraints:",
    len(model.plant_inventory_balance)
)

assert len(model.plant_inventory_balance) == 12

print("\nSTEP 8C: PASSED")


PLANT INVENTORY CONSTRAINTS CREATED
Plant inventory balance constraints: 12

STEP 8C: PASSED


In [18]:

# STEP 9B: Add transportation capacity constraints

import pyomo.environ as pyo

# Shipments cannot exceed the weekly capacity
# of their assigned transportation lane.

def transport_capacity_rule(m, l, t):

    return (
        m.ship[l, t]
        <= m.lane_capacity[l]
    )

model.transport_capacity_limit = pyo.Constraint(
    model.L,
    model.T,
    rule=transport_capacity_rule
)

print("TRANSPORTATION CAPACITY CONSTRAINTS CREATED")
print("=" * 45)

print(
    "Transportation capacity constraints:",
    len(model.transport_capacity_limit)
)

assert len(model.transport_capacity_limit) == 48

print("\nSTEP 9B: PASSED")


TRANSPORTATION CAPACITY CONSTRAINTS CREATED
Transportation capacity constraints: 48

STEP 9B: PASSED


In [19]:
# STEP 9C: Restrict shipments beyond planning horizon

last_week = max(weeks)

def transport_horizon_rule(m, l, t):

    lead_time = int(
        pyo.value(m.lane_lead_time[l])
    )

    arrival_week = t + lead_time

    # Do not allow shipments arriving
    # after the final planning week.
    if arrival_week > last_week:
        return m.ship[l, t] == 0

    return pyo.Constraint.Skip

model.transport_horizon_limit = pyo.Constraint(
    model.L,
    model.T,
    rule=transport_horizon_rule
)

print("TRANSPORTATION HORIZON CONSTRAINTS CREATED")
print("=" * 45)

print(
    "Late-shipment restrictions:",
    len(model.transport_horizon_limit)
)

assert len(model.transport_horizon_limit) == 4

print("\nSTEP 9C: PASSED")


TRANSPORTATION HORIZON CONSTRAINTS CREATED
Late-shipment restrictions: 4

STEP 9C: PASSED


In [20]:
# STEP 9D: Add distribution center inventory balances

first_week = min(weeks)

# Identify which transportation lanes serve each DC
lanes_by_dc = {
    dc: [
        lane for lane in lanes
        if lane_destination[lane] == dc
    ]
    for dc in distribution_centers
}

# Validate every DC has transportation options
assert all(
    len(lanes_by_dc[dc]) > 0
    for dc in distribution_centers
)

# Inventory balance with transportation lead times
def dc_inventory_balance_rule(m, d, t):

    # 1. Beginning inventory
    if t == first_week:
        beginning = initial_dc_inventory[d]
    else:
        beginning = m.dc_inventory[d, t - 1]

    # 2. Shipments arriving during this week
    arrivals = 0

    for lane in lanes_by_dc[d]:

        lead_time = int(
            pyo.value(m.lane_lead_time[lane])
        )

        departure_week = t - lead_time

        if departure_week in m.T:
            arrivals += m.ship[lane, departure_week]

    # 3. Regional customer demand
    customer_demand = m.demand[d, t]

    # 4. Inventory conservation
    return (
        beginning
        + arrivals
        - customer_demand
        == m.dc_inventory[d, t]
    )

model.dc_inventory_balance = pyo.Constraint(
    model.D,
    model.T,
    rule=dc_inventory_balance_rule
)

print("DC INVENTORY BALANCE CONSTRAINTS CREATED")
print("=" * 45)

print(
    "DC inventory balance constraints:",
    len(model.dc_inventory_balance)
)

assert len(model.dc_inventory_balance) == 24

print("\nSTEP 9D: PASSED")


DC INVENTORY BALANCE CONSTRAINTS CREATED
DC inventory balance constraints: 24

STEP 9D: PASSED


In [ ]:
# STEP 9E: Add DC storage constraints (corrected)

import pyomo.environ as pyo

# Remove any partially constructed constraint
# from the previous failed execution.
if hasattr(model, "dc_storage_capacity_limit"):
    model.del_component(model.dc_storage_capacity_limit)

def dc_storage_capacity_rule(m, d, t):

    # 1. Beginning inventory
    if t == first_week:
        beginning = initial_dc_inventory[d]
    else:
        beginning = m.dc_inventory[d, t - 1]

    # 2. Calculate arriving shipments
    arrivals = 0
    has_arrivals = False

    for lane in lanes_by_dc[d]:

        lead_time = int(
            pyo.value(m.lane_lead_time[lane])
        )

        departure_week = t - lead_time

        if departure_week in m.T:
            arrivals += m.ship[lane, departure_week]
            has_arrivals = True

    # 3. Handle Week 1 with no possible arrivals
    if t == first_week and not has_arrivals:

        capacity = pyo.value(m.dc_capacity[d])

        if beginning > capacity:
            raise ValueError(
                f"Starting inventory exceeds capacity: {d}"
            )

        # Known feasible condition; no variable involved
        return pyo.Constraint.Feasible

    # 4. Enforce inventory storage capacity
    return beginning + arrivals <= m.dc_capacity[d]


model.dc_storage_capacity_limit = pyo.Constraint(
    model.D,
    model.T,
    rule=dc_storage_capacity_rule
)

# 5. Validate
assert len(model.dc_storage_capacity_limit) == 24

print("DC STORAGE CAPACITY CONSTRAINTS CREATED")
print("=" * 45)
print(
    "Storage capacity constraints:",
    len(model.dc_storage_capacity_limit)
)
print("Week 1 constant-inventory case: HANDLED")
print("\nSTEP 9E: PASSED")

DC STORAGE CAPACITY CONSTRAINTS CREATED
Storage capacity constraints: 24
Week 1 constant-inventory case: HANDLED

STEP 9E: PASSED


In [23]:
# STEP 9F: Enforce comparable ending inventory

# Retrieve baseline terminal component inventory
terminal_components = (
    baseline_component_inventory.loc[
        baseline_component_inventory["Week"] == last_week
    ].set_index("Component_ID")[
        "Ending_Inventory"
    ].to_dict()
)

# Retrieve baseline terminal plant inventory
terminal_plant_rows = baseline_plant_inventory.loc[
    baseline_plant_inventory["Week"] == last_week,
    "Ending_Inventory"
]

assert len(terminal_plant_rows) == 1

terminal_plant = float(
    terminal_plant_rows.iloc[0]
)

# Retrieve baseline terminal DC inventory
terminal_dcs = (
    baseline_dc_inventory.loc[
        baseline_dc_inventory["Week"] == last_week
    ].set_index("DC_ID")[
        "Ending_Inventory"
    ].to_dict()
)

# Validate baseline terminal inventory records
assert set(terminal_components) == set(model.C)
assert set(terminal_dcs) == set(model.D)

# Component ending inventory constraints
def terminal_component_rule(m, c):

    return (
        m.component_inventory[c, last_week]
        == terminal_components[c]
    )

model.terminal_component_inventory = pyo.Constraint(
    model.C,
    rule=terminal_component_rule
)

# Plant ending inventory constraint
model.terminal_plant_inventory = pyo.Constraint(
    expr=(
        model.plant_inventory[last_week]
        == terminal_plant
    )
)

# Distribution center ending inventory constraints
def terminal_dc_rule(m, d):

    return (
        m.dc_inventory[d, last_week]
        == terminal_dcs[d]
    )

model.terminal_dc_inventory = pyo.Constraint(
    model.D,
    rule=terminal_dc_rule
)

print("TERMINAL INVENTORY CONSTRAINTS CREATED")
print("=" * 45)

print("Components:", len(model.terminal_component_inventory))
print("Plant:", len(model.terminal_plant_inventory))
print("Distribution centers:", len(model.terminal_dc_inventory))

print("\nBaseline ending inventory:")
print("Components:", terminal_components)
print("Plant:", terminal_plant)
print("DCs:", terminal_dcs)

print("\nSTEP 9F: PASSED")


TERMINAL INVENTORY CONSTRAINTS CREATED
Components: 3
Plant: 1
Distribution centers: 2

Baseline ending inventory:
Components: {'ELEC_001': 0, 'BATT_001': 0, 'ENCL_001': 0}
Plant: 0.0
DCs: {'DC_EAST': 21, 'DC_WEST': 24}

STEP 9F: PASSED


In [24]:
# STEP 9G: Final mathematical model validation

from pyomo.repn.standard_repn import generate_standard_repn

# Step 9 constraint counts
constraint_counts = {
    "Transportation Capacity":
        len(model.transport_capacity_limit),

    "Transportation Horizon":
        len(model.transport_horizon_limit),

    "DC Inventory Balance":
        len(model.dc_inventory_balance),

    "DC Storage Capacity":
        len(model.dc_storage_capacity_limit),

    "Terminal Component Inventory":
        len(model.terminal_component_inventory),

    "Terminal Plant Inventory":
        len(model.terminal_plant_inventory),

    "Terminal DC Inventory":
        len(model.terminal_dc_inventory)
}

step9_count = sum(constraint_counts.values())

assert step9_count == 106

# Count all active constraints
active_constraints = list(
    model.component_data_objects(
        pyo.Constraint,
        active=True
    )
)

total_constraints = len(active_constraints)

# Check linearity of every active constraint
for constraint in active_constraints:

    representation = generate_standard_repn(
        constraint.body
    )

    assert representation.is_linear(), (
        f"Nonlinear constraint: {constraint.name}"
    )

# Verify objective remains linear
objective_rep = generate_standard_repn(
    model.total_cost.expr
)

assert objective_rep.is_linear()

# Confirm total expected model structure
assert total_constraints == 247

# Demand and initial inventory checks
assert np.isclose(
    sum(regional_demand.values()),
    6180
)

assert set(initial_dc_inventory) == set(model.D)
assert set(initial_component_inventory) == set(model.C)

print("FINAL OPTIMIZATION MODEL VALIDATION")
print("=" * 50)

print("\nSTEP 9 CONSTRAINT SUMMARY:")
for name, count in constraint_counts.items():
    print(f"{name}: {count}")

print("\nTotal Step 9 constraints:", step9_count)
print("Total model constraints:", total_constraints)

print("\nConstraint linearity: PASSED")
print("Objective linearity: PASSED")
print("Demand parameters: PASSED")
print("Inventory parameters: PASSED")

print("\nSTEP 9 VALIDATION: PASSED")
print("MODEL READY FOR SOLVER")


FINAL OPTIMIZATION MODEL VALIDATION

STEP 9 CONSTRAINT SUMMARY:
Transportation Capacity: 48
Transportation Horizon: 4
DC Inventory Balance: 24
DC Storage Capacity: 24
Terminal Component Inventory: 3
Terminal Plant Inventory: 1
Terminal DC Inventory: 2

Total Step 9 constraints: 106
Total model constraints: 247

Constraint linearity: PASSED
Objective linearity: PASSED
Demand parameters: PASSED
Inventory parameters: PASSED

STEP 9 VALIDATION: PASSED
MODEL READY FOR SOLVER


In [25]:
# STEP 10A: Solve complete supply network model

import pyomo.environ as pyo
from pyomo.opt import SolverStatus, TerminationCondition

# Initialize HiGHS solver
solver = pyo.SolverFactory("highs")

assert solver.available(
    exception_flag=False
), "HiGHS solver is unavailable"

print("SOLVING HARDWARE SUPPLY NETWORK")
print("=" * 50)

# Solve without loading an unverified solution
results = solver.solve(
    model,
    tee=False,
    load_solutions=False
)

solver_status = results.solver.status
termination = results.solver.termination_condition

print("Solver:", "HiGHS")
print("Status:", solver_status)
print("Termination:", termination)

# Only load a verified optimal solution
assert termination == TerminationCondition.optimal, (
    f"Optimization did not reach optimality: {termination}"
)

assert solver_status == SolverStatus.ok, (
    f"Solver returned unexpected status: {solver_status}"
)

model.solutions.load_from(results)

optimized_total_cost = pyo.value(
    model.total_cost
)

print("\nOPTIMIZATION COMPLETED SUCCESSFULLY")
print("=" * 50)
print("Optimal total cost: $",
      round(optimized_total_cost, 2))
print("Solver validation: PASSED")


SOLVING HARDWARE SUPPLY NETWORK
Solver: HiGHS
Status: ok
Termination: optimal

OPTIMIZATION COMPLETED SUCCESSFULLY
Optimal total cost: $ 404131.2
Solver validation: PASSED


In [26]:
# STEP 10B: Extract optimized cost breakdown

import pandas as pd
import numpy as np

optimized_costs = {
    "Procurement": pyo.value(
        model.procurement_cost_expr
    ),
    "Manufacturing": pyo.value(
        model.manufacturing_cost_expr
    ),
    "Transportation": pyo.value(
        model.freight_cost_expr
    ),
    "DC Inventory Holding": pyo.value(
        model.holding_cost_expr
    )
}

optimized_cost_summary = pd.DataFrame([
    {"Cost_Category": category, "Cost_USD": cost}
    for category, cost in optimized_costs.items()
])

# Reconcile category totals to objective
assert np.isclose(
    optimized_cost_summary["Cost_USD"].sum(),
    optimized_total_cost,
    atol=0.01
)

print("OPTIMIZED COST BREAKDOWN")
display(optimized_cost_summary.round(2))

print("\nTotal optimized cost: $",
      round(optimized_total_cost, 2))

print("Cost reconciliation: PASSED")


OPTIMIZED COST BREAKDOWN


,Cost_Category,Cost_USD
0,Procurement,253725.0
1,Manufacturing,124630.0
2,Transportation,25656.5
3,DC Inventory Holding,119.7



Total optimized cost: $ 404131.2
Cost reconciliation: PASSED


In [27]:
# STEP 10C: Compare optimized and baseline costs

# Baseline cost loaded in Step 2
baseline_cost = float(baseline_total_cost)
optimized_cost = float(optimized_total_cost)

# Calculate savings
cost_savings = baseline_cost - optimized_cost

savings_pct = (
    cost_savings / baseline_cost * 100
)

comparison = pd.DataFrame({
    "Metric": [
        "Baseline Cost",
        "Optimized Cost",
        "Cost Savings",
        "Cost Improvement (%)"
    ],
    "Value": [
        baseline_cost,
        optimized_cost,
        cost_savings,
        savings_pct
    ]
})

print("BASELINE VS OPTIMIZATION")
print("=" * 45)

display(comparison.round(2))

# The optimizer should not cost more than a
# feasible baseline under identical constraints
# and accounting conventions.
assert optimized_cost <= baseline_cost + 0.01, (
    "Optimized cost exceeds baseline. "
    "Review model comparability and constraints."
)

print("\nCost comparison: PASSED")
print("Optimization improvement:",
      round(savings_pct, 2), "%")


BASELINE VS OPTIMIZATION


,Metric,Value
0,Baseline Cost,414139.70
1,Optimized Cost,404131.20
2,Cost Savings,10008.50
3,Cost Improvement (%),2.42



Cost comparison: PASSED
Optimization improvement: 2.42 %


In [28]:
# STEP 10D: Extract optimized supply network decisions

import pandas as pd
import numpy as np
import pyomo.environ as pyo

# Convert solved integer variables safely
def integer_value(variable):
    value = float(pyo.value(variable))
    assert np.isfinite(value)
    assert abs(value - round(value)) < 1e-5
    return int(round(value))

# 1. Optimized supplier purchase orders
purchase_rows = []

for s, c in model.SC:
    for t in model.T:
        qty = integer_value(model.purchase[s, c, t])
        lead = int(pyo.value(model.supplier_lead_time[s, c]))
        cost = float(pyo.value(model.supplier_cost[s, c]))

        purchase_rows.append({
            "Order_Week": int(t),
            "Supplier_ID": s,
            "Component_ID": c,
            "Order_Units": qty,
            "Arrival_Week": int(t) + lead,
            "Unit_Cost_USD": cost,
            "Purchase_Cost_USD": qty * cost
        })

optimized_purchases = pd.DataFrame(purchase_rows)

# 2. Optimized weekly manufacturing
production_rows = []

for t in model.T:
    qty = integer_value(model.production[t])

    production_rows.append({
        "Week": int(t),
        "Production_Units": qty,
        "Manufacturing_Cost_USD": qty * assembly_cost
    })

optimized_production = pd.DataFrame(production_rows)

# 3. Optimized transportation decisions
shipment_rows = []

for l in model.L:
    for t in model.T:
        qty = integer_value(model.ship[l, t])
        lead = int(pyo.value(model.lane_lead_time[l]))
        cost = float(pyo.value(model.lane_cost[l]))

        shipment_rows.append({
            "Ship_Week": int(t),
            "Lane_ID": l,
            "DC_ID": lane_destination[l],
            "Ship_Units": qty,
            "Arrival_Week": int(t) + lead,
            "Transit_Time_Weeks": lead,
            "Cost_Per_Unit_USD": cost,
            "Freight_Cost_USD": qty * cost
        })

optimized_shipments = pd.DataFrame(shipment_rows)

# Validate extracted decision tables
assert len(optimized_purchases) == 72
assert len(optimized_production) == 12
assert len(optimized_shipments) == 48

assert optimized_purchases["Order_Units"].ge(0).all()
assert optimized_production["Production_Units"].ge(0).all()
assert optimized_shipments["Ship_Units"].ge(0).all()

print("OPTIMIZED DECISIONS EXTRACTED")
print("=" * 45)

print("Supplier decision records:", len(optimized_purchases))
print("Production records:", len(optimized_production))
print("Transportation records:", len(optimized_shipments))

print("\nOPTIMIZED PRODUCTION:")
display(optimized_production)

print("\nACTIVE SUPPLIER ORDERS:")
display(
    optimized_purchases.loc[
        optimized_purchases["Order_Units"] > 0
    ].head(15)
)

print("\nACTIVE TRANSPORTATION SHIPMENTS:")
display(
    optimized_shipments.loc[
        optimized_shipments["Ship_Units"] > 0
    ].head(15)
)

print("\nSTEP 10D: PASSED")


OPTIMIZED DECISIONS EXTRACTED
Supplier decision records: 72
Production records: 12
Transportation records: 48

OPTIMIZED PRODUCTION:


,Week,Production_Units,Manufacturing_Cost_USD
0,1,580,12760.0
1,2,580,12760.0
2,3,546,12012.0
3,4,580,12760.0
4,5,450,9900.0
5,6,570,12540.0
6,7,525,11550.0
7,8,450,9900.0
8,9,450,9900.0
9,10,450,9900.0



ACTIVE SUPPLIER ORDERS:


,Order_Week,Supplier_ID,Component_ID,Order_Units,Arrival_Week,Unit_Cost_USD,Purchase_Cost_USD
0,1,SUP_A,ELEC_001,450,3,32.0,14400.0
1,2,SUP_A,ELEC_001,450,4,32.0,14400.0
2,3,SUP_A,ELEC_001,450,5,32.0,14400.0
3,4,SUP_A,ELEC_001,450,6,32.0,14400.0
4,5,SUP_A,ELEC_001,450,7,32.0,14400.0
5,6,SUP_A,ELEC_001,450,8,32.0,14400.0
6,7,SUP_A,ELEC_001,450,9,32.0,14400.0
7,8,SUP_A,ELEC_001,450,10,32.0,14400.0
8,9,SUP_A,ELEC_001,450,11,32.0,14400.0
9,10,SUP_A,ELEC_001,34,12,32.0,1088.0



ACTIVE TRANSPORTATION SHIPMENTS:


,Ship_Week,Lane_ID,DC_ID,Ship_Units,Arrival_Week,Transit_Time_Weeks,Cost_Per_Unit_USD,Freight_Cost_USD
0,1,LANE_001,DC_EAST,192,2,1,3.5,672.0
1,2,LANE_001,DC_EAST,278,3,1,3.5,973.0
2,3,LANE_001,DC_EAST,309,4,1,3.5,1081.5
3,4,LANE_001,DC_EAST,309,5,1,3.5,1081.5
4,5,LANE_001,DC_EAST,309,6,1,3.5,1081.5
5,6,LANE_001,DC_EAST,309,7,1,3.5,1081.5
6,7,LANE_001,DC_EAST,309,8,1,3.5,1081.5
7,8,LANE_001,DC_EAST,309,9,1,3.5,1081.5
8,9,LANE_001,DC_EAST,244,10,1,3.5,854.0
9,10,LANE_001,DC_EAST,155,11,1,3.5,542.5



STEP 10D: PASSED


In [29]:
# STEP 10E: Independently validate optimized flows

component_stock = {
    c: int(q)
    for c, q in initial_component_inventory.items()
}

plant_stock = int(initial_plant_inventory)

dc_stock = {
    d: int(q)
    for d, q in initial_dc_inventory.items()
}

component_records = []
plant_records = []
dc_records = []

for week in weeks:

    produced = int(
        optimized_production.loc[
            optimized_production["Week"] == week,
            "Production_Units"
        ].iloc[0]
    )

    # Validate production capacity
    assert produced <= plant_capacity

    # 1. Component inventory
    for c in components:

        beginning = component_stock[c]

        receipts = int(
            optimized_purchases.loc[
                (optimized_purchases["Arrival_Week"] == week)
                & (optimized_purchases["Component_ID"] == c),
                "Order_Units"
            ].sum()
        )

        consumption = int(produced * bom_quantity[c])
        ending = beginning + receipts - consumption

        assert ending >= 0
        assert ending == integer_value(
            model.component_inventory[c, week]
        )

        component_records.append({
            "Week": week,
            "Component_ID": c,
            "Beginning_Inventory": beginning,
            "Supplier_Receipts": receipts,
            "Production_Consumption": consumption,
            "Ending_Inventory": ending
        })

        component_stock[c] = ending

    # 2. Manufacturing plant finished goods
    plant_beginning = plant_stock

    outbound = int(
        optimized_shipments.loc[
            optimized_shipments["Ship_Week"] == week,
            "Ship_Units"
        ].sum()
    )

    plant_ending = plant_beginning + produced - outbound

    assert plant_ending >= 0
    assert plant_ending == integer_value(
        model.plant_inventory[week]
    )

    plant_records.append({
        "Week": week,
        "Beginning_Inventory": plant_beginning,
        "Production_Units": produced,
        "Outbound_Shipments": outbound,
        "Ending_Inventory": plant_ending
    })

    plant_stock = plant_ending

    # 3. Distribution center inventory
    for d in distribution_centers:

        beginning = dc_stock[d]

        arrivals = int(
            optimized_shipments.loc[
                (optimized_shipments["DC_ID"] == d)
                & (optimized_shipments["Arrival_Week"] == week),
                "Ship_Units"
            ].sum()
        )

        demand_units = int(regional_demand[d, week])

        available = beginning + arrivals

        # Storage checked before customer fulfillment
        assert available <= dc_capacity[d], (
            f"DC capacity exceeded: {d}, Week {week}"
        )

        # Full customer fulfillment
        assert available >= demand_units, (
            f"Customer shortage: {d}, Week {week}"
        )

        ending = available - demand_units

        assert ending == integer_value(
            model.dc_inventory[d, week]
        )

        dc_records.append({
            "Week": week,
            "DC_ID": d,
            "Beginning_Inventory": beginning,
            "Shipment_Arrivals": arrivals,
            "Customer_Demand": demand_units,
            "Fulfilled_Units": demand_units,
            "Ending_Inventory": ending
        })

        dc_stock[d] = ending

# Convert simulation records to tables
optimized_component_ledger = pd.DataFrame(component_records)
optimized_plant_ledger = pd.DataFrame(plant_records)
optimized_dc_ledger = pd.DataFrame(dc_records)

# Validate individual supplier and lane capacities
for row in optimized_purchases.itertuples(index=False):
    assert row.Order_Units <= supplier_capacity[
        (row.Supplier_ID, row.Component_ID)
    ]

for row in optimized_shipments.itertuples(index=False):
    assert row.Ship_Units <= lane_capacity[row.Lane_ID]

# Validate final inventory matches baseline
for c in components:
    baseline_end = baseline_component_inventory.loc[
        (baseline_component_inventory["Week"] == last_week)
        & (baseline_component_inventory["Component_ID"] == c),
        "Ending_Inventory"
    ].iloc[0]

    assert np.isclose(component_stock[c], baseline_end)

for d in distribution_centers:
    baseline_end = baseline_dc_inventory.loc[
        (baseline_dc_inventory["Week"] == last_week)
        & (baseline_dc_inventory["DC_ID"] == d),
        "Ending_Inventory"
    ].iloc[0]

    assert np.isclose(dc_stock[d], baseline_end)

baseline_plant_end = baseline_plant_inventory.loc[
    baseline_plant_inventory["Week"] == last_week,
    "Ending_Inventory"
].iloc[0]

assert np.isclose(plant_stock, baseline_plant_end)

# Independently reconstruct modeled cost
recalculated_cost = (
    optimized_purchases["Purchase_Cost_USD"].sum()
    + optimized_production["Manufacturing_Cost_USD"].sum()
    + optimized_shipments["Freight_Cost_USD"].sum()
    + sum(
        dc_holding_cost[d] * ending
        for d, ending in zip(
            optimized_dc_ledger["DC_ID"],
            optimized_dc_ledger["Ending_Inventory"]
        )
    )
)

assert np.isclose(
    recalculated_cost,
    optimized_total_cost,
    atol=0.01
)

assert optimized_dc_ledger["Fulfilled_Units"].sum() == 6180

print("INDEPENDENT OPTIMIZATION VALIDATION")
print("=" * 50)
print("Component inventory balances: PASSED")
print("Manufacturing inventory balances: PASSED")
print("Supplier capacity: PASSED")
print("Transportation capacity: PASSED")
print("Transportation arrivals: PASSED")
print("DC storage capacity: PASSED")
print("Full customer demand fulfillment: PASSED")
print("Terminal inventory reconciliation: PASSED")
print("Independent cost reconciliation: PASSED")

print("\nRecalculated optimized cost: $",
      round(recalculated_cost, 2))

print("\nSTEP 10E: PASSED")


INDEPENDENT OPTIMIZATION VALIDATION
Component inventory balances: PASSED
Manufacturing inventory balances: PASSED
Supplier capacity: PASSED
Transportation capacity: PASSED
Transportation arrivals: PASSED
DC storage capacity: PASSED
Full customer demand fulfillment: PASSED
Terminal inventory reconciliation: PASSED
Independent cost reconciliation: PASSED

Recalculated optimized cost: $ 404131.2

STEP 10E: PASSED


In [30]:
# STEP 10F: Export optimized model results

from pathlib import Path

output_dir = repo_root / "outputs" / "tables"
output_dir.mkdir(parents=True, exist_ok=True)

optimized_outputs = {
    "optimized_purchase_orders.csv": optimized_purchases,
    "optimized_production_plan.csv": optimized_production,
    "optimized_shipments.csv": optimized_shipments,
    "optimized_component_inventory.csv": optimized_component_ledger,
    "optimized_plant_inventory.csv": optimized_plant_ledger,
    "optimized_dc_inventory.csv": optimized_dc_ledger,
    "optimized_cost_breakdown.csv": optimized_cost_summary,
    "baseline_vs_optimized.csv": comparison
}

for filename, dataframe in optimized_outputs.items():

    path = output_dir / filename

    dataframe.to_csv(path, index=False)

    assert path.is_file()

    print("Exported:", filename)

print("\nOPTIMIZATION EXPORT COMPLETE")
print("Files created:", len(optimized_outputs))
print("Output directory:", output_dir)
print("\nSTEP 10F: PASSED")

Exported: optimized_purchase_orders.csv
Exported: optimized_production_plan.csv
Exported: optimized_shipments.csv
Exported: optimized_component_inventory.csv
Exported: optimized_plant_inventory.csv
Exported: optimized_dc_inventory.csv
Exported: optimized_cost_breakdown.csv
Exported: baseline_vs_optimized.csv

OPTIMIZATION EXPORT COMPLETE
Files created: 8
Output directory: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-supply-network-optimizer/outputs/tables

STEP 10F: PASSED
